In [ ]:
import os
import numpy as np
import moabb
from moabb.datasets import Cho2017
from moabb.paradigms import LeftRightImagery
import warnings

# Silence the network logs so your terminal stays clean
warnings.filterwarnings("ignore", module="urllib3")
moabb.set_log_level("ERROR") 

# These indices represent Left/Right symmetrical pairs. 
# but these are standard for 64-ch motor imagery).
SYMMETRIC_PAIRS = [
    (0, 1),   # Fp1, Fp2 (Frontal)
    (4, 5),   # F3, F4   (Frontal)
    (12, 13), # FC3, FC4 (Frontal-Central)
    (22, 23), # C3, C4   (Primary Motor Cortex)
    (32, 33), # CP3, CP4 (Central-Parietal)
    (42, 43), # P3, P4   (Parietal)
    (52, 53)  # O1, O2   (Occipital)
]

def build_gigascience_global(num_subjects=15, save_dir="processed_global"):
    os.makedirs(save_dir, exist_ok=True)
    print("Downloading & Formatting GigaScience")
    
    # 1. MOABB Native Filtering (8-32Hz bandpass at fs=512Hz)
    paradigm = LeftRightImagery(fmin=8.0, fmax=32.0)
    dataset = Cho2017()
    
    # 2. Extract Data
    # X_raw shape: [Trials, 64, 1537]
    X_raw, labels, metadata = paradigm.get_data(dataset=dataset, subjects=list(range(1, num_subjects + 1)))
    
    # Convert labels to Binary (Left=0, Right=1)
    y_int = np.where(labels == 'left_hand', 0, 1)
    print(f"-> Raw Data Downloaded: {X_raw.shape}")
    
    print("\nThe Clean-Pipe Spatial Filter")
    # 3. The Reaction-Time Shift (0.5s - 2.5s)
    # Skip the visual reaction (first 0.5s = 256 steps).
    X_shifted = X_raw[:, :, 256:1281] 
    print(f"-> Time-sliced to 2.0s window: {X_shifted.shape}")
    
    # 4. CAR Filter (Common Average Reference) - VECTORIZED
    # Subtract the mean of all 64 channels at every single timestep
    global_mean = np.mean(X_shifted, axis=1, keepdims=True)
    X_car = X_shifted - global_mean
    print("-> Global noise canceled (CAR applied)")
    
    # 5. Z-Score Normalization & Voltage Gating - VECTORIZED
    # Calculate mean/std across the time axis (axis=2) for each channel, in each trial
    t_mean = np.mean(X_car, axis=2, keepdims=True)
    t_std = np.std(X_car, axis=2, keepdims=True)
    X_norm = (X_car - t_mean) / (t_std + 1e-6)
    
    # Clip blinks/noise
    X_gated = np.clip(X_norm, -3.0, 3.0)
    print("-> Data normalized and voltage-gated at 3.0 sigma.")
    
    # 6. Symmetrical Bipolar Mapping (The Virtual Electrodes)
    print("-> Extracting Symmetrical Virtual Channels")
    X_virtual = []
    for left, right in SYMMETRIC_PAIRS:
        # Subtract Right from Left to amplify lateralization
        diff = X_gated[:, left, :] - X_gated[:, right, :]
        X_virtual.append(diff)
    
    # Convert list of arrays back into a single tensor
    # Transpose moves it from [Pairs, Trials, Time] back to [Trials, Pairs, Time]
    final_X = np.transpose(np.array(X_virtual), (1, 0, 2))
    
    # 7. Save the Master Tensors
    np.save(os.path.join(save_dir, "GLOBAL_X_CLEAN.npy"), final_X)
    np.save(os.path.join(save_dir, "GLOBAL_y.npy"), y_int)

    print(f"\nPHASE 1 COMPLETE")
    print(f"Total Trials: {final_X.shape[0]}")
    print(f"Final Shape:  {final_X.shape} (7 Virtual Channels, 1025 Timesteps)")

if __name__ == "__main__":
    build_gigascience_global(num_subjects=15)

c:\Users\risha\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


-> Raw Data Downloaded: (200, 64, 1537)

The Clean-Pipe Spatial Filter
-> Time-sliced to 2.0s window: (200, 64, 1025)
-> Global noise canceled (CAR applied)
-> Data normalized and voltage-gated at 3.0 sigma.
-> Extracting Symmetrical Virtual Channels

PHASE 1 COMPLETE
Total Trials: 200
Final Shape:  (200, 7, 1025) (7 Virtual Channels, 1025 Timesteps)


In [2]:
import os
import numpy as np
import librosa
from scipy.stats import kurtosis
from scipy.signal import butter, filtfilt
from tqdm.auto import tqdm
import warnings

warnings.filterwarnings("ignore")

#new splitwise butterworth filter
def butter_bandpass_filter(data, lowcut, highcut, fs=512, order=4):
    """
    Slices the signal into specific frequency bands without shifting the phase.
    """
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    # filtfilt applies the filter forward and backward so the wave doesn't distort
    y = filtfilt(b, a, data)
    return y

def extract_acoustic_metrics(signal, sr=512):
    """Helper function to keep the loop clean."""
    rms = np.mean(librosa.feature.rms(y=signal)[0])
    zcr = np.mean(librosa.feature.zero_crossing_rate(y=signal)[0])
    centroid = np.mean(librosa.feature.spectral_centroid(y=signal, sr=sr)[0])
    kurt = kurtosis(signal, fisher=True, bias=False)
    
    mfccs = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=2)
    mfcc_1 = np.mean(mfccs[0])
    mfcc_2 = np.mean(mfccs[1])
    
    return [rms, zcr, centroid, kurt, mfcc_1, mfcc_2]


def build_phase_2_dual_band(data_dir="processed_global", sr=512):
    print("DUAL-BAND ACOUSTIC EXTRACTION")
    
    try:
        X_clean = np.load(os.path.join(data_dir, "GLOBAL_X_CLEAN.npy"))
        y = np.load(os.path.join(data_dir, "GLOBAL_y.npy"))
    except FileNotFoundError:
        print("[ERROR] Could not find Phase 1 output. Run Phase 1 first.")
        return
        
    num_trials, num_virtual_channels, num_timesteps = X_clean.shape
    all_features = []
    
    print("-> Slicing Mu (8-12Hz) & Beta (13-30Hz) tracks and extracting audio textures...")
    
    for trial_idx in tqdm(range(num_trials), desc="Processing Trials"):
        trial_features = []
        
        for ch_idx in range(num_virtual_channels):
            raw_signal = X_clean[trial_idx, ch_idx, :]
            
            # 1. Split the wave into the two biological rhythms
            mu_signal = butter_bandpass_filter(raw_signal, 8.0, 12.0, fs=sr)
            beta_signal = butter_bandpass_filter(raw_signal, 13.0, 30.0, fs=sr)
            
            # 2. Extract Acoustic Textures from BOTH
            mu_metrics = extract_acoustic_metrics(mu_signal, sr)
            beta_metrics = extract_acoustic_metrics(beta_signal, sr)
            
            # 3. Combine them all into one flat array for this trial
            trial_features.extend(mu_metrics)
            trial_features.extend(beta_metrics)
            
        all_features.append(trial_features)
        
    feature_matrix = np.array(all_features)
    
    np.save(os.path.join(data_dir, "GLOBAL_FEATURES_DUAL.npy"), feature_matrix)
    np.save(os.path.join(data_dir, "GLOBAL_y_DUAL.npy"), y) # Save a copy just to be safe
    
    print(f"\nPHASE 2.5 COMPLETE")
    print(f"Final Feature Matrix Shape: {feature_matrix.shape} (Trials, 84 Features)")

if __name__ == "__main__":
    build_phase_2_dual_band()

DUAL-BAND ACOUSTIC EXTRACTION
-> Slicing Mu (8-12Hz) & Beta (13-30Hz) tracks and extracting audio textures...


Processing Trials:  68%|██████▊   | 135/200 [00:13<00:06, 10.11it/s]


KeyboardInterrupt: 

In [ ]:
import os
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
import warnings

# Keep terminal clean
warnings.filterwarnings("ignore")

def run_phase_3(data_dir="processed_global"):
    print("THE FINAL EVALUATION")
    
    # 1. Load the Data
    X = np.load(os.path.join(data_dir, "GLOBAL_FEATURES_DUAL.npy"))
    y = np.load(os.path.join(data_dir, "GLOBAL_y_DUAL.npy"))
    
    print(f"Loaded Feature Matrix: {X.shape}")
    
    # 2. XGBoost Grid Search Setup
    param_grid = {
        'n_estimators': [50, 100, 200],
        'max_depth': [3, 4, 5],
        'learning_rate': [0.01, 0.05, 0.1],
        'subsample': [0.7, 0.8, 0.9],
        'colsample_bytree': [0.5, 0.8, 1.0] # We can use more features now since there are only 42!
    }
    
    xgb_base = XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1
    )
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    print("Running Grid Search")
    grid_search = GridSearchCV(
        estimator=xgb_base, 
        param_grid=param_grid, 
        cv=cv, 
        scoring='accuracy',
        n_jobs=-1,
        verbose=0
    )
    
    grid_search.fit(X, y)
    
    # 3. The Results

    print(f"BEST TRUE AVERAGE ACCURACY: {grid_search.best_score_ * 100:.2f}%")
  
    print("\nOptimal Settings Found:")
    for param, value in grid_search.best_params_.items():
        print(f"   {param}: {value}")
        
    # 4. Feature Importance Mapping
    # Rebuild the names of our 42 features to see what the AI used
    region_pairs = [
        "Fp1-Fp2 (Frontal)", 
        "F3-F4 (Frontal)", 
        "FC3-FC4 (Frontal-Central)", 
        "C3-C4 (Primary Motor)", 
        "CP3-CP4 (Central-Parietal)", 
        "P3-P4 (Parietal)", 
        "O1-O2 (Occipital)"
    ]
    acoustic_metrics = ['RMS Volume', 'ZCR Texture', 'Centroid Pitch', 'Kurtosis', 'MFCC 1', 'MFCC 2']
    
    feature_names = []
    for region in region_pairs:
        for metric in acoustic_metrics:
            feature_names.append(f"{region} | {metric}")
            
    best_model = grid_search.best_estimator_
    importances = best_model.feature_importances_
    indices = np.argsort(importances)[-10:] # Top 10
    
    print("\nTOP 10 BIOLOGICAL FEATURES")
    for i in reversed(indices):
        print(f"{feature_names[i]:<84}: {importances[i]:.4f}")

if __name__ == "__main__":
    run_phase_3()

THE FINAL EVALUATION
Loaded Feature Matrix: (3080, 84)
Running Grid Search


KeyboardInterrupt: 

In [ ]:
import os
import numpy as np
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from tqdm.auto import tqdm
import warnings

warnings.filterwarnings("ignore")

def run_phase_3_dual_showdown(data_dir="processed_global"):
    print("THE DUAL-BAND SHOWDOWN")
    
    # 1. LOAD THE NEW DUAL-BAND DATA
    X = np.load(os.path.join(data_dir, "GLOBAL_FEATURES_DUAL.npy"))
    y = np.load(os.path.join(data_dir, "GLOBAL_y_DUAL.npy"))
    
    # THIS SHOULD PRINT (200, 84) 
    print(f"Loaded Feature Matrix: {X.shape}\n") 
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    # RANDOM FOREST TUNING
    print("Tuning Random Forest")
    rf_grid = {
        'n_estimators': [100, 300],
        'max_depth': [3, 5, 7],
        'max_features': ['sqrt', 'log2'],
        'min_samples_leaf': [1, 3]
    }
    rf_base = RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1)
    rf_search = GridSearchCV(rf_base, rf_grid, cv=cv, scoring='accuracy', n_jobs=-1)
    rf_search.fit(X, y)
    best_rf = rf_search.best_estimator_
    print(f"   RF Best CV Score: {rf_search.best_score_ * 100:.2f}%")

    # XGBOOST TUNING
    print("Tuning XGBoost")
    xgb_grid = {
        'n_estimators': [100, 200],
        'max_depth': [3, 4, 5],
        'learning_rate': [0.01, 0.05, 0.1],
        'subsample': [0.7, 0.9],
        'colsample_bytree': [0.5, 0.8, 1.0]
    }
    xgb_base = XGBClassifier(objective='binary:logistic', eval_metric='logloss', random_state=42, n_jobs=-1)
    xgb_search = GridSearchCV(xgb_base, xgb_grid, cv=cv, scoring='accuracy', n_jobs=-1)
    xgb_search.fit(X, y)
    best_xgb = xgb_search.best_estimator_
    print(f"   XGB Best CV Score: {xgb_search.best_score_ * 100:.2f}%\n")

    # THE 100-RUN MONTE CARLO
    print("Running 100-Iteration Monte Carlo Simulation")
    iterations = 100
    rf_scores = []
    xgb_scores = []
    
    for i in tqdm(range(iterations), desc="Monte Carlo"):
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=i, stratify=y
        )
        
        best_rf.fit(X_train, y_train)
        rf_scores.append(best_rf.score(X_test, y_test))
        
        best_xgb.fit(X_train, y_train)
        xgb_scores.append(best_xgb.score(X_test, y_test))

    rf_scores = np.array(rf_scores)
    xgb_scores = np.array(xgb_scores)

    print("FINAL MONTE CARLO RESULTS (DUAL-BAND)")
    print("\nRANDOM FOREST:")
    print(f"   True Average : {rf_scores.mean() * 100:.2f}%")
    print(f"   Peak Accuracy: {rf_scores.max() * 100:.2f}%")
    print(f"   Lowest Hit   : {rf_scores.min() * 100:.2f}%")
    print("\nXGBOOST:")
    print(f"   True Average : {xgb_scores.mean() * 100:.2f}%")
    print(f"   Peak Accuracy: {xgb_scores.max() * 100:.2f}%")
    print(f"   Lowest Hit   : {xgb_scores.min() * 100:.2f}%")

    #DUAL-BAND FEATURE IMPORTANCE
    region_pairs = [
        "Fp1-Fp2 (Frontal)", 
        "F3-F4 (Frontal)", 
        "FC3-FC4 (Frontal-Central)", 
        "C3-C4 (Primary Motor)", 
        "CP3-CP4 (Central-Parietal)", 
        "P3-P4 (Parietal)", 
        "O1-O2 (Occipital)"
    ]
    bands = ["Mu (8-12Hz)", "Beta (13-30Hz)"]
    acoustic_metrics = ['RMS Volume', 'ZCR Texture', 'Centroid Pitch', 'Kurtosis', 'MFCC 1', 'MFCC 2']
    
    # Build the exact 84-feature name list in the order they were extracted
    feature_names = []
    for region in region_pairs:
        for band in bands:
            for metric in acoustic_metrics:
                feature_names.append(f"{region} | {band} | {metric}")
                
    overall_best = best_xgb if xgb_scores.mean() > rf_scores.mean() else best_rf
    importances = overall_best.feature_importances_
    indices = np.argsort(importances)[-10:]
    
    winner_name = "XGBoost" if xgb_scores.mean() > rf_scores.mean() else "Random Forest"
    print(f"--- TOP 10 BIOLOGICAL FEATURES ({winner_name}) ---")
    for i in reversed(indices):
        print(f"{feature_names[i]:<50}: {importances[i]:.4f}")

if __name__ == "__main__":
    run_phase_3_dual_showdown()

THE DUAL-BAND SHOWDOWN
Loaded Feature Matrix: (3080, 84)

Tuning Random Forest
   RF Best CV Score: 53.67%
Tuning XGBoost
   XGB Best CV Score: 54.51%

Running 100-Iteration Monte Carlo Simulation


Monte Carlo: 100%|██████████| 100/100 [02:07<00:00,  1.28s/it]

FINAL MONTE CARLO RESULTS (DUAL-BAND)

RANDOM FOREST:
   True Average : 52.54%
   Peak Accuracy: 56.82%
   Lowest Hit   : 48.21%

XGBOOST:
   True Average : 53.05%
   Peak Accuracy: 58.12%
   Lowest Hit   : 48.38%
--- TOP 10 BIOLOGICAL FEATURES (XGBoost) ---
FC3-FC4 (Frontal-Central) | Mu (8-12Hz) | RMS Volume: 0.0163
CP3-CP4 (Central-Parietal) | Mu (8-12Hz) | Kurtosis: 0.0153
F3-F4 (Frontal) | Beta (13-30Hz) | Centroid Pitch : 0.0152
O1-O2 (Occipital) | Mu (8-12Hz) | ZCR Texture     : 0.0150
O1-O2 (Occipital) | Mu (8-12Hz) | Centroid Pitch  : 0.0148
F3-F4 (Frontal) | Mu (8-12Hz) | Kurtosis          : 0.0145
FC3-FC4 (Frontal-Central) | Beta (13-30Hz) | Centroid Pitch: 0.0142
FC3-FC4 (Frontal-Central) | Mu (8-12Hz) | Kurtosis: 0.0139
Fp1-Fp2 (Frontal) | Mu (8-12Hz) | MFCC 1          : 0.0138
CP3-CP4 (Central-Parietal) | Beta (13-30Hz) | RMS Volume: 0.0137


In [3]:
import os
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
import moabb
from moabb.datasets import Cho2017
from moabb.paradigms import LeftRightImagery
from scipy.signal import butter, filtfilt
from scipy.stats import kurtosis
import librosa
from tqdm.auto import tqdm
import warnings

warnings.filterwarnings("ignore")
moabb.set_log_level("ERROR")

# Use your exact same Phase 1 pairs
SYMMETRIC_PAIRS = [(0, 1), (4, 5), (12, 13), (22, 23), (32, 33), (42, 43), (52, 53)]

def process_single_subject(subject_id):
    # --- 1. DATALOADER ---
    paradigm = LeftRightImagery(fmin=8.0, fmax=32.0)
    dataset = Cho2017()
    X_raw, labels, _ = paradigm.get_data(dataset=dataset, subjects=[subject_id])
    y = np.where(labels == 'left_hand', 0, 1)
    
    # --- 2. CLEAN-PIPE ---
    X = X_raw[:, :, 256:1281] # Slicing
    X = X - np.mean(X, axis=1, keepdims=True) # CAR
    t_mean = np.mean(X, axis=2, keepdims=True)
    t_std = np.std(X, axis=2, keepdims=True)
    X = np.clip((X - t_mean) / (t_std + 1e-6), -3.0, 3.0) # Z-Score & Gate
    
    # Virtual Channels
    X_virt = np.transpose(np.array([X[:, l, :] - X[:, r, :] for l, r in SYMMETRIC_PAIRS]), (1, 0, 2))
    
    # --- 3. DUAL-BAND ACOUSTIC EXTRACTION ---
    features = []
    for trial_idx in range(X_virt.shape[0]):
        trial_feat = []
        for ch_idx in range(X_virt.shape[1]):
            sig = X_virt[trial_idx, ch_idx, :]
            
            # Butterworth slices
            mu = filtfilt(*butter(4, [8.0/256.0, 12.0/256.0], btype='band'), sig)
            beta = filtfilt(*butter(4, [13.0/256.0, 30.0/256.0], btype='band'), sig)
            
            for band_sig in [mu, beta]:
                rms = np.mean(librosa.feature.rms(y=band_sig)[0])
                zcr = np.mean(librosa.feature.zero_crossing_rate(y=band_sig)[0])
                cent = np.mean(librosa.feature.spectral_centroid(y=band_sig, sr=512)[0])
                kurt = kurtosis(band_sig, fisher=True, bias=False)
                mfcc = librosa.feature.mfcc(y=band_sig, sr=512, n_mfcc=2)
                trial_feat.extend([rms, zcr, cent, kurt, np.mean(mfcc[0]), np.mean(mfcc[1])])
                
        features.append(trial_feat)
        
    return np.array(features), y

def run_subject_specific_validation(num_subjects=15):
    print(f"--- RUNNING WITHIN-SUBJECT VALIDATION ({num_subjects} Subjects) ---")
    
    xgb_model = XGBClassifier(
        n_estimators=100, max_depth=4, learning_rate=0.05, 
        subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=42
    )
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    all_scores = []
    
    # Loop through each person individually
    for sub in tqdm(range(1, num_subjects + 1), desc="Evaluating Subjects"):
        X_feat, y = process_single_subject(sub)
        
        # Test the model purely on this person's brainwaves
        scores = cross_val_score(xgb_model, X_feat, y, cv=cv, scoring='accuracy')
        sub_mean = scores.mean() * 100
        all_scores.append(sub_mean)
        
        print(f"   Subject {sub:02d} Accuracy: {sub_mean:.2f}%")
        
    print(f"GRAND AVERAGE (Subject-Specific): {np.mean(all_scores):.2f}%")
    print(f"PEAK SUBJECT ACCURACY: {np.max(all_scores):.2f}% (Subject {np.argmax(all_scores)+1})")
    print(f"LOWEST SUBJECT ACCURACY: {np.min(all_scores):.2f}%")

if __name__ == "__main__":
    run_subject_specific_validation(num_subjects=15)

--- RUNNING WITHIN-SUBJECT VALIDATION (15 Subjects) ---


Evaluating Subjects:   7%|▋         | 1/15 [00:23<05:32, 23.75s/it]

   Subject 01 Accuracy: 66.00%


Evaluating Subjects:  13%|█▎        | 2/15 [00:47<05:06, 23.56s/it]

   Subject 02 Accuracy: 50.00%


...

   Subject 13 Accuracy: 69.00%


Evaluating Subjects:  93%|█████████▎| 14/15 [05:34<00:23, 23.44s/it]

   Subject 14 Accuracy: 78.50%


Evaluating Subjects: 100%|██████████| 15/15 [05:57<00:00, 23.82s/it]

   Subject 15 Accuracy: 46.00%
GRAND AVERAGE (Subject-Specific): 55.38%
PEAK SUBJECT ACCURACY: 78.50% (Subject 14)
LOWEST SUBJECT ACCURACY: 43.00%


In [6]:
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
import moabb
from moabb.datasets import Cho2017
from moabb.paradigms import LeftRightImagery
from scipy.signal import butter, filtfilt
from scipy.stats import kurtosis
import librosa
import warnings

warnings.filterwarnings("ignore")
moabb.set_log_level("ERROR")

# Focus purely on Subject 14
SUBJECT = 1
SYMMETRIC_PAIRS = [(0, 1), (4, 5), (12, 13), (22, 23), (32, 33), (42, 43), (52, 53)]

def extract_features_from_chunk(signal):
    """Helper to extract our 6 acoustic features"""
    rms = np.mean(librosa.feature.rms(y=signal)[0])
    zcr = np.mean(librosa.feature.zero_crossing_rate(y=signal)[0])
    cent = np.mean(librosa.feature.spectral_centroid(y=signal, sr=512)[0])
    kurt = kurtosis(signal, fisher=True, bias=False)
    mfcc = librosa.feature.mfcc(y=signal, sr=512, n_mfcc=2)
    return [rms, zcr, cent, kurt, np.mean(mfcc[0]), np.mean(mfcc[1])]

print(f" Subject {SUBJECT} specific tuning")

# 1. Load Subject 14
paradigm = LeftRightImagery(fmin=8.0, fmax=32.0)
dataset = Cho2017()
X_raw, labels, _ = paradigm.get_data(dataset=dataset, subjects=[SUBJECT])
y = np.where(labels == 'left_hand', 0, 1)

# 2. Clean-Pipe
X = X_raw[:, :, 256:1281] # The 2.0s window
X = X - np.mean(X, axis=1, keepdims=True) # CAR
X = np.clip((X - np.mean(X, axis=2, keepdims=True)) / (np.std(X, axis=2, keepdims=True) + 1e-6), -3.0, 3.0)
X_virt = np.transpose(np.array([X[:, l, :] - X[:, r, :] for l, r in SYMMETRIC_PAIRS]), (1, 0, 2))

# 3. Temporal Micro-Slicing & Dual-Band Extraction
print("Running Temporal Micro-Slicing...")
features = []
# We split the 1025 timesteps into 3 chunks: [0-400], [300-700], [600-1025]
# This captures the Start, Middle, and End of the thought.
time_chunks = [(0, 400), (300, 700), (600, 1025)] 

for trial_idx in range(X_virt.shape[0]):
    trial_feat = []
    for ch_idx in range(X_virt.shape[1]):
        full_sig = X_virt[trial_idx, ch_idx, :]
        
        # Split into Mu and Beta
        mu = filtfilt(*butter(4, [8.0/256.0, 12.0/256.0], btype='band'), full_sig)
        beta = filtfilt(*butter(4, [13.0/256.0, 30.0/256.0], btype='band'), full_sig)
        
        # Extract features for EACH time chunk, for EACH band
        for start, end in time_chunks:
            trial_feat.extend(extract_features_from_chunk(mu[start:end]))
            trial_feat.extend(extract_features_from_chunk(beta[start:end]))
            
    features.append(trial_feat)

X_feat = np.array(features)
print(f"New High-Def Feature Matrix: {X_feat.shape} (Includes Timeline Dynamics)")

# 4. Evaluation
xgb_model = XGBClassifier(
    n_estimators=150, max_depth=3, learning_rate=0.03, 
    subsample=0.7, colsample_bytree=0.5, n_jobs=-1, random_state=42
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(xgb_model, X_feat, y, cv=cv, scoring='accuracy')
print(f"Subject {SUBJECT} Specific Accuracy: {scores.mean() * 100:.2f}%")
print(f"   Peak Fold: {scores.max() * 100:.2f}%")

 Subject 1 specific tuning
Running Temporal Micro-Slicing...
New High-Def Feature Matrix: (200, 252) (Includes Timeline Dynamics)
Subject 1 Specific Accuracy: 66.50%
   Peak Fold: 75.00%
